# Ironman Race Results Analysis

This notebook demonstrates how to scrape, clean, analyze, and visualize Ironman race results. The workflow includes data extraction, cleaning, transformation, analysis, and visualization of performance trends.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
import re
import os
# Load all sheets from the Excel file into separate DataFrames

def load_ironman_sheets(excel_path):
    """
    Load all sheets from the Ironman Excel file into a dictionary of DataFrames.
    Drops the sheet named 'hiddensheet' if present.
    Returns: dict of {sheet_name: DataFrame}
    """
    all_sheets = pd.read_excel(excel_path, sheet_name=None)
    if 'hiddenSheet' in all_sheets:
        print("Dropping sheet: hiddenSheet")
        all_sheets.pop('hiddenSheet')
    for name, df in all_sheets.items():
        print(f"Loaded sheet: {name} (rows: {len(df)})")
    return all_sheets

# Usage example:
im_excel_path = '../data/Ironman_RaceResults.xlsx'
im_sheets = load_ironman_sheets(im_excel_path)


imhalf_excel_path = '../data/Ironman70.3_RaceResults.xlsx'
imhalf_sheets = load_ironman_sheets(imhalf_excel_path)
imhalf_sheets.keys()

In [ ]:
from database import get_engine
from api_handling import fetch_athlete_country_search, fetch_program_ids

engine = get_engine()
athlete = pd.read_sql_table('athlete', engine)
race_results = pd.read_sql_table('race_results', engine)
events = pd.read_sql_table('events', engine)

other_events_path = '../docs/other_event_data.csv'
other_df = pd.read_csv(other_events_path)
event_ids = other_df['event_id'].unique()

#event_id = 184360
#event_name = "2024 World Triathlon Age-Group Championships Torremolinos-Andalucia"
#progs = fetch_program_ids(event_id)

filtered_results = race_results[race_results['event_id'].isin(event_ids)]
events_unique = events.drop_duplicates(subset=['event_id'])

# Merge event_name into filtered_results
filtered_results = filtered_results.merge(
    events_unique[['event_id', 'event_name']],
    on='event_id',
    how='left'
)

filtered_results = filtered_results.merge(
    athlete[['full_name', 'gender']],
    left_on='athlete_full_name', right_on='full_name',
    how='left'
)

filtered_results = filtered_results[['athlete_full_name', 'total_time', 'event_name', 'gender']]
filtered_results = filtered_results.rename(columns={
    'athlete_full_name': 'Contact',
    'event_name': 'Event',
    'total_time': 'Finish Time Formatted',
    'gender': 'Gender'
})

filtered_results

In [ ]:
# Add Gender and Finish Time (Seconds) Columns

def parse_time_to_seconds(time_str):
    """Convert a time string (e.g., '8:15:30') to seconds."""
    if pd.isnull(time_str):
        return np.nan
    parts = str(time_str).split(':')
    if len(parts) == 3:
        h, m, s = parts
    elif len(parts) == 2:
        h, m, s = 0, parts[0], parts[1]
    else:
        return np.nan
    try:
        return int(h) * 3600 + int(m) * 60 + float(s)
    except Exception:
        return np.nan
    
def extract_year(event_str):
    """Extracts a 4-digit year from the event string."""
    match = re.search(r'(20\d{2})', str(event_str))
    if match:
        return int(match.group(1))
    return np.nan

def infer_gender_from_age_group(age_group):
    """Infer gender from Age Group code (F=Female, M=Male, else Unknown)."""
    if isinstance(age_group, str):
        if age_group.startswith('F'):
            return 'Female'
        elif age_group.startswith('M'):
            return 'Male'
    return 'Unknown'

def add_columns_to_sheets(sheets):
    for sheet_name, df in sheets.items():
        if 'Finish Time Formatted' in df.columns:
            df['FinishSeconds'] = df['Finish Time Formatted'].apply(parse_time_to_seconds)
        if 'Age Group' in df.columns:
            df['Gender'] = df['Age Group'].apply(infer_gender_from_age_group)
        if 'Event' in df.columns:
            df['Year'] = df['Event'].apply(extract_year)
        if {'FinishSeconds', 'Gender', 'Event', 'Year'}.issubset(df.columns):
            # compute minimum finish time per event, year, and gender
            min_times = df.groupby(['Event', 'Year', 'Gender'])['FinishSeconds'].transform('min')
            mask = df['FinishSeconds'] <= min_times * 1.10
            df = df.loc[mask].copy()
            sheets[sheet_name] = df

add_columns_to_sheets(im_sheets)
add_columns_to_sheets(imhalf_sheets)

filtered_results['FinishSeconds'] = filtered_results['Finish Time Formatted'].apply(parse_time_to_seconds)
filtered_results['Year'] = filtered_results['Event'].apply(extract_year)

combined_df = pd.concat(
    list(im_sheets.values()) + list(imhalf_sheets.values()) + [filtered_results],
    ignore_index=True
)

name_country_dict = dict(
    athlete
    .drop_duplicates(subset=['full_name'])[['full_name','country']]
    .values
)
combined_df['Country'] = combined_df['Contact'].map(name_country_dict)
combined_df['Gender'] = combined_df['Gender'].str.title()

missing_names = combined_df.loc[combined_df['Country'].isna(), 'Contact'].unique()
for name in missing_names:
    try:
        country = fetch_athlete_country_search(name)
    except Exception as e:
        #print(f"Error fetching country for {name}: {e}")
        country = None
    combined_df.loc[combined_df['Contact'] == name, 'Country'] = country

# Count how many Country entries are null
null_count = combined_df['Country'].isna().sum()
print(f"Number of missing country values: {null_count}")


In [ ]:
combined_df[combined_df['Event']== '2024 T100 Triathlon World Tour at CLASH Endurance Miami']

In [ ]:
# Calculate number of finishers and capture contacts within 4%–10% of the winner

def get_finishers(df, gender, percent):
    df_g = df[df['Gender'] == gender].copy()
    if df_g.empty or 'FinishSeconds' not in df_g.columns:
        return [], 0
    winner = df_g['FinishSeconds'].min()
    if pd.isnull(winner):
        return [], 0
    cutoff = winner * (1 + percent/100)
    mask = (
        (df_g['FinishSeconds'] <= cutoff) &
        ((df_g['Country'] == 'United States') | df_g['Country'].isna())
    )
    quals = df_g.loc[mask, 'Contact'].unique().tolist()
    return quals, len(quals)

def get_finishers_by_event(df):
    results = []
    percentiles = list(range(4, 11))
    for event in df['Event'].dropna().unique():
        df_e = df[df['Event'] == event]
        for yr in sorted(df_e['Year'].dropna().unique()):
            df_y = df_e[df_e['Year'] == yr]
            for g in ['Male','Female']:
                for pct in percentiles:
                    contacts, cnt = get_finishers(df_y, g, pct)
                    results.append({
                        'Event': event,
                        'Year': int(yr),
                        'Gender': g,
                        'Percent': pct,
                        'NumFinishers': cnt,
                        'Contacts': contacts
                    })
    return pd.DataFrame(results)

# Run it on the combined_df:
results_df = get_finishers_by_event(combined_df)

In [ ]:
# Visualize and summarize by event, year, and gender
for (event, year) in results_df[['Event', 'Year']].drop_duplicates().itertuples(index=False):
    display_df = results_df[(results_df['Event'] == event) & (results_df['Year'] == year)]\
        .pivot(index='Percent', columns='Gender', values='NumFinishers')
    
    winner_info = {}
    for gender in ['Male', 'Female']:
        mask = (
            (combined_df['Event'] == event) &
            (combined_df['Year'] == year) &
            (combined_df['Gender'] == gender)
        )
        df_event_gender = combined_df[mask]
        if not df_event_gender.empty and 'FinishSeconds' in df_event_gender.columns:
            winner_row = df_event_gender.loc[df_event_gender['FinishSeconds'].idxmin()]
            winner_name = winner_row['Contact']
            winning_time = winner_row['Finish Time Formatted']
            winner_info[gender] = (winner_name, winning_time)
        else:
            winner_info[gender] = ('N/A', 'N/A')

    print(f"\nSummary Table for {event} ({year})")
    for gender in ['Male', 'Female']:
        print(f"{gender} Winner: {winner_info[gender][0]}, Winning Time: {winner_info[gender][1]}")
    display(display_df)
    plt.figure(figsize=(6, 4))
    title_str = (f"Finishers within X% of Winner - {event} ({year})\n"
                 f"Male Winner: {winner_info['Male'][0]} ({winner_info['Male'][1]}) | "
                 f"Female Winner: {winner_info['Female'][0]} ({winner_info['Female'][1]})")
    sns.heatmap(display_df, annot=True, fmt='d', cmap='Blues')
    plt.title(title_str)
    plt.ylabel('Percent Over Winner')
    plt.xlabel('Gender')
    plt.show()

In [ ]:
b_events = [
    '2023 World Triathlon Long Distance Championships Ibiza',
    '2024 World Triathlon Long Distance Championships Townsville',
    '2025 World Triathlon Long Distance Championships Pontevedra',
    '2024 T100 Triathlon World Tour at CLASH Endurance Miami',
    '2024 T100 Triathlon World Tour Grand Final Dubai',
    '2024 T100 Triathlon World Tour Ibiza',
    '2024 T100 Triathlon World Tour Lake Las Vegas',
    '2024 T100 Triathlon World Tour London',
    '2024 T100 Triathlon World Tour San Francisco',
    '2024 T100 Triathlon World Tour Singapore',
    '2025 T100 Triathlon World Tour San Francisco',
    '2025 T100 Triathlon World Tour Singapore',
    '2025 T100 Triathlon World Tour Vancouver',
    'IRONMAN Texas',
    'IRONMAN 70.3 St George'
]
c_events = ['IRONMAN Arizona',
            'IRONMAN Lake Placid',
            'IRONMAN Chattanooga',
            'IRONMAN 70.3 Eagleman',
            'IRONMAN 70.3 Oceanside',
            'IRONMAN 70.3 Boulder',
            'IRONMAN 70.3 Chattanooga'
]
d_events = [
    '2024 IRONMAN 70.3 Indian Wells La Quinta',
    'IRONMAN 70.3 Oregon',
    'IRONMAN 70.3 Santa Cruz',
    'IRONMAN 70.3 Texas', 
    'IRONMAN 70.3 Maine',
    'IRONMAN 70.3 Wisconsin',
]

In [ ]:
def get_event_category(event_name):
    """Determine if event is b, c, or d category"""
    for e in b_events:
        if e in event_name:
            return 'b'
    for e in c_events:
        if e in event_name:
            return 'c'
    for e in d_events:
        if e in event_name:
            return 'd'
    return None

def count_qualifiers_with_thresholds(results_df, year, b_thresh, c_thresh, d_thresh):
    """Count unique qualifiers for a given year and threshold combination"""
    year_data = results_df[results_df['Year'] == year].copy()
    year_data['EventCategory'] = year_data['Event'].apply(get_event_category)
    
    # Filter out events without category
    year_data = year_data[year_data['EventCategory'].notna()]
    
    qualified_athletes = {'Male': set(), 'Female': set()}
    
    for _, row in year_data.iterrows():
        event_cat = row['EventCategory']
        percent = row['Percent']
        gender = row['Gender']
        contacts = row['Contacts']
        
        # Determine required threshold for this event
        if event_cat == 'b':
            required_thresh = b_thresh
        elif event_cat == 'c':
            required_thresh = c_thresh
        elif event_cat == 'd':
            required_thresh = d_thresh
        else:
            continue
            
        # If this row matches the required threshold, add contacts
        if percent == required_thresh:
            qualified_athletes[gender].update(contacts)
    
    return len(qualified_athletes['Male']), len(qualified_athletes['Female'])

# Chart 1: B-Events Dynamic (C=6%, D=4% fixed)
thresholds = list(range(4, 11))
chart1_data = []

for year in [2023, 2024]:
    for thresh in thresholds:
        male_count, female_count = count_qualifiers_with_thresholds(
            results_df, year, b_thresh=thresh, c_thresh=6, d_thresh=4
        )
        chart1_data.append({
            'Year': year,
            'Threshold': thresh,
            'Male': male_count,
            'Female': female_count,
        })

chart1_df = pd.DataFrame(chart1_data)

# Visualization for Chart 1
for year in [2023, 2024]:
    year_data = chart1_df[chart1_df['Year'] == year]
    
    plt.figure(figsize=(10, 6))
    plt.plot(year_data['Threshold'], year_data['Male'], 'o-', label='Male', color='blue', linewidth=2)
    plt.plot(year_data['Threshold'], year_data['Female'], 'o-', label='Female', color='magenta', linewidth=2)
    
    plt.xlabel('B-Event Threshold (%)')
    plt.ylabel('Number of Qualifiers')
    plt.title(f'Qualifier Sensitivity to B-Event Threshold - {year}\n(C-Events: 6%, D-Events: 4%)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(thresholds)
    plt.show()

In [ ]:
# Overlay line chart for B, C, D event dynamic threshold analysis (Male/Female counts)
thresholds = list(range(4, 11))
chart1_data = []
chart2_data = []
chart3_data = []

for year in [2023, 2024]:
    for thresh in thresholds:
        # Chart 1: B dynamic
        male_count_b, female_count_b = count_qualifiers_with_thresholds(
            results_df, year, b_thresh=thresh, c_thresh=6, d_thresh=4
        )
        chart1_data.append({
            'Year': year,
            'Threshold': thresh,
            'Male_B': male_count_b,
            'Female_B': female_count_b,
        })
        # Chart 2: C dynamic
        male_count_c, female_count_c = count_qualifiers_with_thresholds(
            results_df, year, b_thresh=8, c_thresh=thresh, d_thresh=4
        )
        chart2_data.append({
            'Year': year,
            'Threshold': thresh,
            'Male_C': male_count_c,
            'Female_C': female_count_c,
        })
        # Chart 3: D dynamic
        male_count_d, female_count_d = count_qualifiers_with_thresholds(
            results_df, year, b_thresh=8, c_thresh=6, d_thresh=thresh
        )
        chart3_data.append({
            'Year': year,
            'Threshold': thresh,
            'Male_D': male_count_d,
            'Female_D': female_count_d,
        })

chart1_df = pd.DataFrame(chart1_data)
chart2_df = pd.DataFrame(chart2_data)
chart3_df = pd.DataFrame(chart3_data)

# Visualization: Overlay all lines for comparison
for year in [2023, 2024]:
    plt.figure(figsize=(12, 8))
    # Chart 1: B dynamic
    year_data1 = chart1_df[chart1_df['Year'] == year]
    plt.plot(year_data1['Threshold'], year_data1['Male_B'], 'o-', label='Male (B Events)', color='blue', linewidth=2)
    plt.plot(year_data1['Threshold'], year_data1['Female_B'], 'o-', label='Female (B Events)', color='magenta', linewidth=2)
    # Chart 2: C dynamic
    year_data2 = chart2_df[chart2_df['Year'] == year]
    plt.plot(year_data2['Threshold'], year_data2['Male_C'], 's--', label='Male (C Events)', color='navy', linewidth=2)
    plt.plot(year_data2['Threshold'], year_data2['Female_C'], 's--', label='Female (C Events)', color='deeppink', linewidth=2)
    # Chart 3: D dynamic
    year_data3 = chart3_df[chart3_df['Year'] == year]
    plt.plot(year_data3['Threshold'], year_data3['Male_D'], 'd:', label='Male (D Events)', color='cyan', linewidth=2)
    plt.plot(year_data3['Threshold'], year_data3['Female_D'], 'd:', label='Female (D Events)', color='orange', linewidth=2)

    plt.xlabel('Threshold (%)', fontsize=12)
    plt.ylabel('Number of Qualifiers', fontsize=12)
    plt.title(f'Qualifier Sensitivity to Thresholds - {year}\n(B, C, D Events)', fontsize=14)
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.xticks(thresholds)
    plt.tight_layout()
    plt.show()

In [ ]:
# Export the last chart to clipboard for easy pasting into PowerPoint
# Note: This requires the 'imageio' and 'Pillow' libraries and works on Windows
# Run this cell after generating a chart to copy it to the clipboard

import io
from PIL import ImageGrab

def copy_last_figure_to_clipboard():
    buf = io.BytesIO()
    plt.gcf().savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    ImageGrab.ImageGrab().clipboard_clear()
    ImageGrab.ImageGrab().clipboard_append(img)
    print("Chart copied to clipboard! You can now paste it into PowerPoint or Word.")

# Usage: After running a chart cell, run this cell to copy the last chart to clipboard
# copy_last_figure_to_clipboard()